# Stage 12: Realistic Backtesting

This notebook studies turnover, transaction costs, threshold rebalancing, and cost-adjusted portfolio performance.

In [1]:
from __future__ import annotations

from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

project_root = Path.cwd().resolve()
if project_root.name == '12_realistic_backtesting':
    project_root = project_root.parents[1]

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.backtesting import RollingBacktester
from src.backtesting.transaction_costs import TransactionCostModel
from src.benchmarks import run_strategy_comparison, build_performance_comparison_table
from src.dashboard.plots import (
    plot_cost_adjusted_comparison,
    plot_drawdown_curves,
    plot_transaction_costs,
    plot_turnover_series,
)
from src.optimization import HERCAllocator, HRPAllocator


## 1. Load Data

In [2]:
rng = np.random.default_rng(2050)
dates = pd.date_range(start='2021-01-01', periods=504, freq='B')
common_factor = rng.normal(0.0002, 0.0045, size=(len(dates), 1))
idiosyncratic = rng.normal(
    loc=0.0003,
    scale=np.array([0.009, 0.012, 0.015, 0.010, 0.013]),
    size=(len(dates), 5),
)
returns_df = pd.DataFrame(
    common_factor + idiosyncratic,
    index=dates,
    columns=['Equity', 'IT', 'Gold', 'Bonds', 'Energy'],
)
returns_df.head()


,Equity,IT,Gold,Bonds,Energy
2021-01-01,-0.000022,-0.008593,-0.007470,-0.014963,0.002662
2021-01-04,0.025703,0.012766,0.010843,0.014956,-0.004284
2021-01-05,-0.014985,0.000535,0.010738,0.005701,-0.022806
2021-01-06,0.005180,0.004227,-0.018446,0.016627,0.014215
2021-01-07,0.004352,0.013430,-0.019779,-0.006115,0.017097


## 2. Run Calendar Monthly Backtest

In [3]:
cost_model = TransactionCostModel(base_bps=10.0, slippage_bps=5.0)
calendar_hrp = RollingBacktester(
    allocator=HRPAllocator(covariance_method='ledoit_wolf'),
    train_window=252,
    rebalance_frequency='M',
    rebalance_mode='calendar',
    transaction_cost_model=cost_model,
).run(returns_df)
calendar_hrp['performance_metrics']


{'cumulative_return': 0.12844060364370113,
 'cagr': 0.12844060364370113,
 'sharpe': 1.0357564796876375,
 'sortino': 1.1994847311959178,
 'volatility': 0.10242804858795317,
 'max_drawdown': -0.04996660401816333,
 'final_value': 1127384.9379300596,
 'transaction_cost': 988.626792782343,
 'total_transaction_cost': 988.626792782343,
 'total_turnover': 0.6239351805858542,
 'average_turnover': 0.04799501389121955,
 'number_of_rebalances': 13}

## 3. Run Threshold Rebalancing Backtest

In [4]:
threshold_hrp = RollingBacktester(
    allocator=HRPAllocator(covariance_method='ledoit_wolf'),
    train_window=252,
    rebalance_frequency='M',
    rebalance_mode='threshold',
    threshold=0.05,
    transaction_cost_model=cost_model,
).run(returns_df)
threshold_hrp['performance_metrics']


{'cumulative_return': 0.11943181702267402,
 'cagr': 0.11943181702267402,
 'sharpe': 0.9624076672325753,
 'sortino': 1.1343502815817972,
 'volatility': 0.10183977535067779,
 'max_drawdown': -0.050758770200035386,
 'final_value': 1118469.7092743535,
 'transaction_cost': 890.0553346039972,
 'total_transaction_cost': 890.0553346039972,
 'total_turnover': 0.5731892060226428,
 'average_turnover': 0.07164865075283035,
 'number_of_rebalances': 8}

## 4. Compare Turnover

In [5]:
calendar_turnover = pd.Series(
    calendar_hrp['rebalance_log']['turnover'].values,
    index=pd.to_datetime(calendar_hrp['rebalance_log']['rebalance_date']),
    name='Calendar Turnover',
) if not calendar_hrp['rebalance_log'].empty else pd.Series(dtype=float)

threshold_turnover = pd.Series(
    threshold_hrp['rebalance_log']['turnover'].values,
    index=pd.to_datetime(threshold_hrp['rebalance_log']['rebalance_date']),
    name='Threshold Turnover',
) if not threshold_hrp['rebalance_log'].empty else pd.Series(dtype=float)

if not calendar_turnover.empty:
    plot_turnover_series(calendar_turnover).show()
if not threshold_turnover.empty:
    plot_turnover_series(threshold_turnover).show()


## 5. Compare Transaction Costs

In [6]:
if not calendar_hrp['rebalance_log'].empty:
    plot_transaction_costs(calendar_hrp['rebalance_log']).show()
if not threshold_hrp['rebalance_log'].empty:
    plot_transaction_costs(threshold_hrp['rebalance_log']).show()

pd.DataFrame(
    {
        'Calendar': calendar_hrp['turnover_summary'],
        'Threshold': threshold_hrp['turnover_summary'],
    }
)


,Calendar,Threshold
total_turnover,0.623935,0.573189
average_turnover,0.047995,0.071649
max_turnover,0.092252,0.081597
num_rebalances,13.000000,8.000000


## 6. Compare Final Values

In [7]:
plot_cost_adjusted_comparison(
    calendar_hrp['gross_portfolio_values'],
    calendar_hrp['portfolio_values'],
).show()

pd.DataFrame(
    {
        'Calendar HRP': calendar_hrp['performance_metrics'],
        'Threshold HRP': threshold_hrp['performance_metrics'],
    }
)


,Calendar HRP,Threshold HRP
cumulative_return,1.284406e-01,1.194318e-01
cagr,1.284406e-01,1.194318e-01
sharpe,1.035756e+00,9.624077e-01
sortino,1.199485e+00,1.134350e+00
volatility,1.024280e-01,1.018398e-01
max_drawdown,-4.996660e-02,-5.075877e-02
final_value,1.127385e+06,1.118470e+06
transaction_cost,9.886268e+02,8.900553e+02
total_transaction_cost,9.886268e+02,8.900553e+02
total_turnover,6.239352e-01,5.731892e-01


## 7. Compare Drawdowns

In [8]:
plot_drawdown_curves(
    {
        'Calendar HRP': calendar_hrp['drawdown'],
        'Threshold HRP': threshold_hrp['drawdown'],
    }
).show()


## 8. Interpret Results

In [9]:
threshold_scenarios = {}
for threshold in [0.03, 0.05, 0.10]:
    result = RollingBacktester(
        allocator=HERCAllocator(covariance_method='ledoit_wolf'),
        train_window=252,
        rebalance_frequency='M',
        rebalance_mode='threshold',
        threshold=threshold,
        transaction_cost_model=cost_model,
    ).run(returns_df)
    threshold_scenarios[f'HERC threshold {threshold:.0%}'] = result['performance_metrics']

scenario_df = pd.DataFrame(threshold_scenarios).T
display(scenario_df.round(4))

discussion = [
    '### Research Questions',
    f"- HRP total turnover (calendar): `{calendar_hrp['turnover_summary']['total_turnover']:.4f}`",
    f"- HRP total turnover (threshold): `{threshold_hrp['turnover_summary']['total_turnover']:.4f}`",
    f"- HRP total transaction cost (calendar): `{calendar_hrp['performance_metrics']['total_transaction_cost']:.2f}`",
    f"- HRP total transaction cost (threshold): `{threshold_hrp['performance_metrics']['total_transaction_cost']:.2f}`",
    '- Threshold rebalancing can reduce turnover when drift tolerances absorb small allocation changes.',
    '- The gross vs net portfolio value gap provides a direct cost-drag estimate.',
    '- Comparing 3%, 5%, and 10% thresholds helps quantify the tradeoff between responsiveness and trading friction.',
]
display(Markdown('\n'.join(discussion)))


,cumulative_return,cagr,sharpe,sortino,volatility,max_drawdown,final_value,transaction_cost,total_transaction_cost,total_turnover,average_turnover,number_of_rebalances
HERC threshold 3%,0.0994,0.0994,0.7243,0.9314,0.1119,-0.0870,1.088228e+06,10683.4843,10683.4843,6.8374,0.1221,56.0
HERC threshold 5%,0.0978,0.0978,0.7122,0.9180,0.1117,-0.0884,1.086655e+06,10628.0941,10628.0941,6.8071,0.1238,55.0
HERC threshold 10%,0.1139,0.1139,0.8587,1.0466,0.1093,-0.0781,1.106594e+06,6907.8366,6907.8366,4.3823,0.1565,28.0


### Research Questions
- HRP total turnover (calendar): `0.6239`
- HRP total turnover (threshold): `0.5732`
- HRP total transaction cost (calendar): `988.63`
- HRP total transaction cost (threshold): `890.06`
- Threshold rebalancing can reduce turnover when drift tolerances absorb small allocation changes.
- The gross vs net portfolio value gap provides a direct cost-drag estimate.
- Comparing 3%, 5%, and 10% thresholds helps quantify the tradeoff between responsiveness and trading friction.